# MNIST Digit Classification with Backpropagation
A NumPy implementation of a fully connected neural network trained on MNIST. Two experiments compare training settings.

## Experiment 1 — Baseline
128 hidden neurons, learning rate 0.1, batch size 64, 25 epochs.


In [ ]:
import gzip
import struct
import numpy as np
import matplotlib.pyplot as plt


# --------------------------------------------------
# 1. Load MNIST data
# --------------------------------------------------

def load_images(filename):
    with gzip.open(filename, "rb") as f:
        magic, count, rows, cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError("Invalid MNIST image file")

        images = np.frombuffer(f.read(), dtype=np.uint8)

    images = images.reshape(count, rows * cols)
    return images.astype(np.float64) / 255.0


def load_labels(filename):
    with gzip.open(filename, "rb") as f:
        magic, count = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError("Invalid MNIST label file")

        labels = np.frombuffer(f.read(), dtype=np.uint8)

    return labels


from pathlib import Path

data_dir = Path("data")

X_train = load_images(data_dir / "train-images-idx3-ubyte.gz")
y_train = load_labels(data_dir / "train-labels-idx1-ubyte.gz")
X_test = load_images(data_dir / "t10k-images-idx3-ubyte.gz")
y_test = load_labels(data_dir / "t10k-labels-idx1-ubyte.gz")

print("Training images:", X_train.shape)
print("Testing images:", X_test.shape)


In [ ]:
# --------------------------------------------------
# 2. Convert labels into one-hot vectors
# --------------------------------------------------

def one_hot(labels):
    result = np.zeros((len(labels), 10))
    result[np.arange(len(labels)), labels] = 1
    return result


Y_train = one_hot(y_train)
Y_test = one_hot(y_test)


In [ ]:

# --------------------------------------------------
# 3. Activation function
# --------------------------------------------------

def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x))


In [ ]:

# --------------------------------------------------
# 4. Initialize neural network
# --------------------------------------------------

np.random.seed(42)

input_size = 784
hidden_size = 128
output_size = 10

learning_rate = 0.1
epochs = 25
batch_size = 64

# Xavier initialization
W1 = np.random.randn(input_size, hidden_size) * np.sqrt(
    2.0 / (input_size + hidden_size)
)

b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * np.sqrt(
    2.0 / (hidden_size + output_size)
)

b2 = np.zeros((1, output_size))


In [ ]:
# --------------------------------------------------
# 5. Forward propagation
# --------------------------------------------------

def forward(X):
    hidden_input = X @ W1 + b1
    hidden_output = sigmoid(hidden_input)

    final_input = hidden_output @ W2 + b2
    final_output = sigmoid(final_input)

    return hidden_output, final_output


In [ ]:
# --------------------------------------------------
# 6. Calculate MSE and classification errors
# --------------------------------------------------

def evaluate(X, Y, labels):
    total_squared_error = 0.0
    total_errors = 0

    # Evaluate in batches to reduce memory usage
    for start in range(0, len(X), 1000):
        end = start + 1000

        _, predictions = forward(X[start:end])

        differences = predictions - Y[start:end]
        total_squared_error += np.sum(differences ** 2)

        predicted_labels = np.argmax(predictions, axis=1)

        total_errors += np.sum(
            predicted_labels != labels[start:end]
        )

    mse = total_squared_error / (len(X) * output_size)

    accuracy = 100.0 * (1 - total_errors / len(X))

    return mse, int(total_errors), accuracy


In [ ]:

# --------------------------------------------------
# 7. Training using backpropagation
# --------------------------------------------------

train_mse_history = []
test_mse_history = []

train_error_history = []
test_error_history = []

for epoch in range(epochs):

    # Shuffle training data before every epoch
    indices = np.random.permutation(len(X_train))

    X_shuffled = X_train[indices]
    Y_shuffled = Y_train[indices]

    for start in range(0, len(X_train), batch_size):
        end = start + batch_size

        X_batch = X_shuffled[start:end]
        Y_batch = Y_shuffled[start:end]

        # Forward propagation
        hidden_output, final_output = forward(X_batch)

        # Backpropagation at output layer
        output_delta = (
            (final_output - Y_batch)
            * final_output
            * (1 - final_output)
        )

        # Backpropagation at hidden layer
        hidden_delta = (
            (output_delta @ W2.T)
            * hidden_output
            * (1 - hidden_output)
        )

        # Calculate gradients
        m = len(X_batch)

        dW2 = hidden_output.T @ output_delta / m
        db2 = np.mean(output_delta, axis=0, keepdims=True)

        dW1 = X_batch.T @ hidden_delta / m
        db1 = np.mean(hidden_delta, axis=0, keepdims=True)

        # Update weights and biases
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

    # Evaluate after each epoch
    train_mse, train_errors, train_accuracy = evaluate(
        X_train, Y_train, y_train
    )

    test_mse, test_errors, test_accuracy = evaluate(
        X_test, Y_test, y_test
    )

    train_mse_history.append(train_mse)
    test_mse_history.append(test_mse)

    train_error_history.append(train_errors)
    test_error_history.append(test_errors)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Accuracy: {train_accuracy:.2f}% | "
        f"Test Accuracy: {test_accuracy:.2f}% | "
        f"Train MSE: {train_mse:.5f} | "
        f"Test MSE: {test_mse:.5f}"
    )

In [ ]:

# --------------------------------------------------
# 8. Plot classification errors
# --------------------------------------------------

epoch_numbers = range(1, epochs + 1)

plt.figure(figsize=(9, 5))

plt.plot(
    epoch_numbers,
    train_error_history,
    label="Training Errors"
)

plt.plot(
    epoch_numbers,
    test_error_history,
    label="Test Errors"
)

plt.xlabel("Epoch Number")
plt.ylabel("Number of Classification Errors")
plt.title("Epoch vs Classification Errors")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("classification_errors.png", dpi=300)
plt.show()


In [ ]:
# --------------------------------------------------
# 9. Plot Mean Squared Error
# --------------------------------------------------

plt.figure(figsize=(9, 5))

plt.plot(
    epoch_numbers,
    train_mse_history,
    label="Training MSE"
)

plt.plot(
    epoch_numbers,
    test_mse_history,
    label="Test MSE"
)

plt.xlabel("Epoch Number")
plt.ylabel("Mean Squared Error")
plt.title("Epoch vs Mean Squared Error")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("mse_plot.png", dpi=300)
plt.show()



In [ ]:
# --------------------------------------------------
# 10. Final results
# --------------------------------------------------

final_train_mse, final_train_errors, final_train_accuracy = evaluate(
    X_train, Y_train, y_train
)

final_test_mse, final_test_errors, final_test_accuracy = evaluate(
    X_test, Y_test, y_test
)

print("\nFINAL RESULTS")
print("------------------------------")
print(f"Training Accuracy: {final_train_accuracy:.2f}%")
print(f"Testing Accuracy: {final_test_accuracy:.2f}%")
print(f"Training Errors: {final_train_errors}")
print(f"Testing Errors: {final_test_errors}")
print(f"Training MSE: {final_train_mse:.6f}")
print(f"Testing MSE: {final_test_mse:.6f}")


## Experiment 2 — Higher learning rate and smaller batches
128 hidden neurons, learning rate 0.5, batch size 32, 40 epochs.


In [ ]:
import gzip
import struct
import numpy as np
import matplotlib.pyplot as plt


# --------------------------------------------------
# 1. Load MNIST data
# --------------------------------------------------

def load_images(filename):
    with gzip.open(filename, "rb") as f:
        magic, count, rows, cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError("Invalid MNIST image file")

        images = np.frombuffer(f.read(), dtype=np.uint8)

    images = images.reshape(count, rows * cols)
    return images.astype(np.float64) / 255.0


def load_labels(filename):
    with gzip.open(filename, "rb") as f:
        magic, count = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError("Invalid MNIST label file")

        labels = np.frombuffer(f.read(), dtype=np.uint8)

    return labels


from pathlib import Path

data_dir = Path("data")

X_train = load_images(data_dir / "train-images-idx3-ubyte.gz")
y_train = load_labels(data_dir / "train-labels-idx1-ubyte.gz")

X_test = load_images(data_dir / "t10k-images-idx3-ubyte.gz")
y_test = load_labels(data_dir / "t10k-labels-idx1-ubyte.gz")

print("Training images:", X_train.shape)
print("Testing images:", X_test.shape)



In [ ]:
# --------------------------------------------------
# 2. Convert labels into one-hot vectors
# --------------------------------------------------

def one_hot(labels):
    result = np.zeros((len(labels), 10))
    result[np.arange(len(labels)), labels] = 1
    return result


Y_train = one_hot(y_train)
Y_test = one_hot(y_test)


In [ ]:
# --------------------------------------------------
# 3. Activation function
# --------------------------------------------------

def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x))

In [ ]:
# --------------------------------------------------
# 4. Initialize neural network
# --------------------------------------------------

np.random.seed(42)

# Experiment 2
input_size = 784
hidden_size = 128
output_size = 10

learning_rate = 0.5
epochs = 40
batch_size = 32

# Xavier initialization
W1 = np.random.randn(input_size, hidden_size) * np.sqrt(
    2.0 / (input_size + hidden_size)
)

b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * np.sqrt(
    2.0 / (hidden_size + output_size)
)

b2 = np.zeros((1, output_size))

In [ ]:
# --------------------------------------------------
# 5. Forward propagation
# --------------------------------------------------

def forward(X):
    hidden_input = X @ W1 + b1
    hidden_output = sigmoid(hidden_input)

    final_input = hidden_output @ W2 + b2
    final_output = sigmoid(final_input)

    return hidden_output, final_output


In [ ]:

# --------------------------------------------------
# 6. Calculate MSE and classification errors
# --------------------------------------------------

def evaluate(X, Y, labels):
    total_squared_error = 0.0
    total_errors = 0

    # Evaluate in batches to reduce memory usage
    for start in range(0, len(X), 1000):
        end = start + 1000

        _, predictions = forward(X[start:end])

        differences = predictions - Y[start:end]
        total_squared_error += np.sum(differences ** 2)

        predicted_labels = np.argmax(predictions, axis=1)

        total_errors += np.sum(
            predicted_labels != labels[start:end]
        )

    mse = total_squared_error / (len(X) * output_size)

    accuracy = 100.0 * (1 - total_errors / len(X))

    return mse, int(total_errors), accuracy



In [ ]:
# --------------------------------------------------
# 7. Training using backpropagation
# --------------------------------------------------

train_mse_history = []
test_mse_history = []

train_error_history = []
test_error_history = []

for epoch in range(epochs):

    # Shuffle training data before every epoch
    indices = np.random.permutation(len(X_train))

    X_shuffled = X_train[indices]
    Y_shuffled = Y_train[indices]

    for start in range(0, len(X_train), batch_size):
        end = start + batch_size

        X_batch = X_shuffled[start:end]
        Y_batch = Y_shuffled[start:end]

        # Forward propagation
        hidden_output, final_output = forward(X_batch)

        # Backpropagation at output layer
        output_delta = (
            (final_output - Y_batch)
            * final_output
            * (1 - final_output)
        )

        # Backpropagation at hidden layer
        hidden_delta = (
            (output_delta @ W2.T)
            * hidden_output
            * (1 - hidden_output)
        )

        # Calculate gradients
        m = len(X_batch)

        dW2 = hidden_output.T @ output_delta / m
        db2 = np.mean(output_delta, axis=0, keepdims=True)

        dW1 = X_batch.T @ hidden_delta / m
        db1 = np.mean(hidden_delta, axis=0, keepdims=True)

        # Update weights and biases
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

    # Evaluate after each epoch
    train_mse, train_errors, train_accuracy = evaluate(
        X_train, Y_train, y_train
    )

    test_mse, test_errors, test_accuracy = evaluate(
        X_test, Y_test, y_test
    )

    train_mse_history.append(train_mse)
    test_mse_history.append(test_mse)

    train_error_history.append(train_errors)
    test_error_history.append(test_errors)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Accuracy: {train_accuracy:.2f}% | "
        f"Test Accuracy: {test_accuracy:.2f}% | "
        f"Train MSE: {train_mse:.5f}| "
        f"Test MSE: {test_mse:.5f}"
    )

In [ ]:
# --------------------------------------------------
# 8. Plot classification errors
# --------------------------------------------------

epoch_numbers = range(1, epochs + 1)

plt.figure(figsize=(9, 5))

plt.plot(
    epoch_numbers,
    train_error_history,
    label="Training Errors"
)

plt.plot(
    epoch_numbers,
    test_error_history,
    label="Test Errors"
)

plt.xlabel("Epoch Number")
plt.ylabel("Number of Classification Errors")
plt.title("Epoch vs Classification Errors")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("classification_errors.png", dpi=300)
plt.show()

In [ ]:
# --------------------------------------------------
# 9. Plot Mean Squared Error
# --------------------------------------------------

plt.figure(figsize=(9, 5))

plt.plot(
    epoch_numbers,
    train_mse_history,
    label="Training MSE"
)

plt.plot(
    epoch_numbers,
    test_mse_history,
    label="Test MSE"
)

plt.xlabel("Epoch Number")
plt.ylabel("Mean Squared Error")
plt.title("Epoch vs Mean Squared Error")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("mse_plot.png", dpi=300)
plt.show()


In [ ]:
# --------------------------------------------------
# 10. Final results
# --------------------------------------------------

final_train_mse, final_train_errors, final_train_accuracy = evaluate(
    X_train, Y_train, y_train
)

final_test_mse, final_test_errors, final_test_accuracy = evaluate(
    X_test, Y_test, y_test
)

print("\nFINAL RESULTS")
print("------------------------------")
print(f"Training Accuracy: {final_train_accuracy:.2f}%")
print(f"Testing Accuracy: {final_test_accuracy:.2f}%")
print(f"Training Errors: {final_train_errors}")
print(f"Testing Errors: {final_test_errors}")
print(f"Training MSE: {final_train_mse:.6f}")
print(f"Testing MSE: {final_test_mse:.6f}")
